# 01 · Preparazione dei dati

Scarica i dataset, crea le liste di file per training e test (in `manifests/`) e controlla che l'audio ci sia.
Ogni cella si può rilanciare: quello che è già stato scaricato o estratto viene saltato.

| Dataset | Uso | Dove finisce |
|---|---|---|
| ASVspoof 5 | train, dev, test | `data/asvspoof5` |
| ASVspoof 2019 LA | train, dev, test | `data/asvspoof2019` |
| ASVspoof 2021 LA e DF | solo test (modelli allenati su 2019 LA) | `data/asvspoof2021` |
| In-the-Wild | solo test | `data/in_the_wild` |
| MUSAN e RIR | augmentation "rumore" | `data/musan`, `data/RIRS_NOISES` |

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))   # per importare la cartella tesi/ del progetto

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tesi import percorsi, dati, augmentation

print("Dati in:", percorsi.DATI)

## 1. ASVspoof 5

Da Hugging Face (`jungjee/asvspoof5`). Sul fisso è già tutto scaricato, quindi la cella non fa niente.

Le liste di train, dev ed eval sono le stesse clip dei primi esperimenti (10.000 di train, 5.000 di dev, 10.000 di eval,
bilanciate, seed 42, durata 1-15 s), così i risultati restano confrontabili. In più c'è l'eval completo (circa 680.000 clip).

In [ ]:
for split in ("train", "dev", "eval"):
    dati.scarica_asvspoof5(split)

In [ ]:
dati.salva_lista(dati.lista_da_manifest("asvspoof5_train_n5000_seed42_dur1-15.csv", "train"), "asvspoof5_train")
dati.salva_lista(dati.lista_da_manifest("asvspoof5_dev_n2500_seed42_dur1-15.csv", "dev"), "asvspoof5_dev")
dati.salva_lista(dati.lista_da_manifest("asvspoof5_eval_n5000_seed42_dur1-15.csv", "eval"), "asvspoof5_eval")
dati.salva_lista(dati.lista_asvspoof5("eval"), "asvspoof5_eval_completo")

## 2. ASVspoof 2019 LA

Da Edinburgh DataShare: https://datashare.ed.ac.uk/handle/10283/3336 (file `LA.zip`, circa 7 GB).
Se il download dalla cella non funziona, scaricalo dal browser e mettilo in `data/asvspoof2019/`.

Si usa intero, con le partizioni ufficiali. È sbilanciato (circa 1 bonafide ogni 9 spoof):
la focal loss usa pesi calcolati dal numero di clip di ogni classe (vedi `tesi/training.py`).

In [ ]:
zip_2019 = percorsi.ASVSPOOF2019 / "LA.zip"
dati.scarica("https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip", zip_2019)
dati.estrai(zip_2019, percorsi.ASVSPOOF2019)

In [ ]:
for split in ("train", "dev", "eval"):
    dati.salva_lista(dati.lista_asvspoof2019(split), f"asvspoof2019_{split}")

## 3. ASVspoof 2021 LA e DF (solo eval)

Il 2021 non ha un train suo: si testano i modelli allenati su 2019 LA, come nei paper.
Audio da Zenodo (LA: https://zenodo.org/records/4837263, DF: https://zenodo.org/records/4835108, in 4 parti),
chiavi con le label da https://www.asvspoof.org/index2021.html.
Il DF pesa circa 35 GB. Se un link non va, scarica a mano e metti i file in `data/asvspoof2021/`.

Oltre alle liste complete ci sono due sottoinsiemi da 10.000 clip (5.000 per classe), per i test più veloci.

In [ ]:
file_2021 = {
    "ASVspoof2021_LA_eval.tar.gz": "https://zenodo.org/records/4837263/files/ASVspoof2021_LA_eval.tar.gz",
    "LA-keys-full.tar.gz": "https://www.asvspoof.org/asvspoof2021/LA-keys-full.tar.gz",
    "DF-keys-full.tar.gz": "https://www.asvspoof.org/asvspoof2021/DF-keys-full.tar.gz",
}
for parte in range(4):
    nome = f"ASVspoof2021_DF_eval_part0{parte}.tar.gz"
    file_2021[nome] = f"https://zenodo.org/records/4835108/files/{nome}"

for nome, url in file_2021.items():
    archivio = dati.scarica(url, percorsi.ASVSPOOF2021 / nome)
    dati.estrai(archivio, percorsi.ASVSPOOF2021)

In [ ]:
for traccia in ("LA", "DF"):
    lista = dati.lista_asvspoof2021(traccia)
    dati.salva_lista(lista, f"asvspoof2021{traccia.lower()}_eval")
    dati.salva_lista(dati.sottoinsieme(lista, 5000), f"asvspoof2021{traccia.lower()}_eval_10k")

## 4. In-the-Wild

Da https://deepfake-total.com/in_the_wild: scarica `release_in_the_wild.zip` dal browser e mettilo in `data/in_the_wild/`.
Sono 31.779 clip di personaggi pubblici, usate solo per il test.

In [ ]:
dati.estrai(percorsi.IN_THE_WILD / "release_in_the_wild.zip", percorsi.IN_THE_WILD)
dati.salva_lista(dati.lista_in_the_wild(), "inthewild")

## 5. MUSAN e RIR (per l'augmentation "rumore")

Da OpenSLR: MUSAN (https://www.openslr.org/17, circa 11 GB) e RIR (https://www.openslr.org/28).

In [ ]:
for url in ("https://www.openslr.org/resources/17/musan.tar.gz",
            "https://www.openslr.org/resources/28/rirs_noises.zip"):
    archivio = dati.scarica(url, percorsi.DATI / url.split("/")[-1])
    dati.estrai(archivio, percorsi.DATI)

## 6. Controllo delle liste

Per ogni lista: numero di clip per classe e file mancanti (devono essere 0).
Sulle liste grandi il controllo dei file richiede qualche minuto.

In [ ]:
righe = []
for file in sorted(percorsi.LISTE.glob("*.csv")):
    if "_seed42_" in file.name:   # manifest dei primi esperimenti, non sono liste
        continue
    lista = dati.carica_lista(file.stem)
    righe.append({"lista": file.stem, "clip": len(lista),
                  "bonafide": (lista["label"] == "bonafide").sum(), "spoof": (lista["label"] == "spoof").sum(),
                  "file mancanti": dati.file_mancanti(lista)})
pd.DataFrame(righe)

## 7. Le augmentation su una clip

Prima un controllo che ffmpeg funzioni con i tre codec, poi lo spettrogramma della stessa clip con ogni augmentation
(figura per il capitolo degli esperimenti, salvata in `artifacts/figure/`).

In [ ]:
ffmpeg = augmentation.trova_ffmpeg()
prova = np.random.default_rng(0).normal(0, 0.1, 16000).astype(np.float32)
for codec, (_, _, bitrate) in augmentation.Codec.CODEC.items():
    print(codec, bitrate[0], "kbit/s:", augmentation.comprimi(prova, codec, bitrate[0], ffmpeg).shape)

In [ ]:
lista = dati.carica_lista("asvspoof5_train")
wave = dati.leggi_audio(lista[lista["label"] == "bonafide"]["percorso"].iloc[0])[:4 * dati.FREQ]

rng = np.random.default_rng(0)
rumore = augmentation.RumoreRiverbero()
versioni = {
    "Originale": wave,
    "RawBoost": augmentation.RawBoost().trasforma(wave, rng),
    "Rumore (MUSAN)": rumore.aggiungi_musan(wave, rng, "noise"),
    "Riverbero (RIR simulata)": rumore.aggiungi_riverbero(wave, rng),
    "Codec (MP3 a 16 kbit/s)": augmentation.comprimi(wave, "mp3", 16, ffmpeg),
}

fig, assi = plt.subplots(len(versioni), 1, figsize=(8, 10), sharex=True)
for ax, (titolo, w) in zip(assi, versioni.items()):
    ax.specgram(w, Fs=dati.FREQ, NFFT=512, noverlap=384, cmap="magma")
    ax.set_title(titolo)
    ax.set_ylabel("Frequenza (Hz)")
assi[-1].set_xlabel("Tempo (s)")
fig.tight_layout()
(percorsi.PROGETTO / "artifacts" / "figure").mkdir(parents=True, exist_ok=True)
fig.savefig(percorsi.PROGETTO / "artifacts" / "figure" / "augmentation.png", dpi=200)